<a href="https://colab.research.google.com/github/hungvuongminh-cell/AI-Testing-Learning/blob/main/Phase_01/0003_a_basic_evaluation_with_AI_as_a_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain-groq

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# 1. Cấu hình API Key (Thay bằng key của bạn)
os.environ["GROQ_API_KEY"] = userdata.get('groq_api')

# 2. Danh sách Model
MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "gemma2-9b-it",
    "mixtral-8x7b-32768",
    "deepseek-r1-distill-llama-70b"
]

MODEL_INDEX = 1

# 3. Khởi tạo Model
model = ChatGroq(model=MODELS[MODEL_INDEX], temperature=0.2)

# 4. MASTER PROMPT MẪU (Sức khỏe bình thường nhưng cấu trúc cực xịn)
# Đây là phần bạn có thể dùng để test khả năng tư duy của các model khác nhau.
master_prompt = """
- **Role**: Bạn là một Chuyên gia Tư vấn Sức khỏe Chủ động (Health Coach) với tư duy y khoa dựa trên bằng chứng (Evidence-based).

- **Task**: Xây dựng một lộ trình cải thiện chất lượng giấc ngủ và giảm căng thẳng cho nhân viên văn phòng.

- **Context**: Tôi là một lập trình viên 30 tuổi, thường xuyên làm việc với máy tính 10 tiếng/ngày, uống 3 ly cafe mỗi ngày và thường xuyên bị khó ngủ, tỉnh giấc lúc 3 giờ sáng. Tôi muốn cải thiện sức khỏe mà không dùng đến thuốc ngủ.

- **Instructions**:
    1. Giải thích cơ chế khoa học của việc tỉnh giấc giữa đêm dựa trên thói quen tiêu thụ caffeine.
    2. Đề xuất 3 bài tập vận động nhẹ nhàng tại chỗ có thể thực hiện trong giờ làm việc.
    3. Thiết kế quy trình "Sleep Hygiene" (vệ sinh giấc ngủ) trong 2 tiếng trước khi đi ngủ.

- **Constraints (Avoiding Bias & Hallucination)**:
    - Chỉ đưa ra các lời khuyên dựa trên các nghiên cứu y khoa đã được công nhận (như từ Mayo Clinic hoặc Harvard Health).
    - Tuyệt đối không được kê đơn thuốc hoặc chẩn đoán bệnh lý thay cho bác sĩ.
    - Nếu có phương pháp nào mang tính thực nghiệm, phải nêu rõ "Cần tham khảo ý kiến chuyên gia".
    - Không được tự bịa ra các số liệu thống kê không có thật.

- **Response Style**: Trình bày khoa học, dễ hiểu, sử dụng Bullet points để các bước hành động rõ ràng. Giọng văn khích lệ nhưng phải khách quan và nghiêm túc.
"""

# 5. Thiết lập Prompt Template (Dùng Master Prompt làm nội dung User gửi đi)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Bạn là một trợ lý AI hữu ích."),
    ("user", master_prompt)
])

# 6. Chạy Chain
chain = prompt_template | model | StrOutputParser()

print(f"--- ĐANG TEST VỚI MODEL: {MODELS[MODEL_INDEX]} ---\n")
ai_response = chain.invoke({})
print(ai_response)
print('\n'*3)

# Giám khảo
# 1. Chúng ta dùng Temperature = 0 để đảm bảo sự khắt khe và nhất quán khi chấm điểm
judge_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

# 2. Nội dung cần chấm (Lấy từ biến 'ai_response' của bước chạy test trước đó)
# Giả sử biến 'i_response' đang chứa câu trả lời của model bạn vừa test
content_to_grade = ai_response

# 3. Thiết lập hệ thống chấm điểm (Rubric)
judge_system_prompt = """
Bạn là một Giám khảo Đánh giá AI chuyên nghiệp (AI Quality Auditor).
Nhiệm vụ của bạn là chấm điểm câu trả lời của một AI khác dựa trên các tiêu chí nghiêm ngặt.

### TIÊU CHÍ CHẤM ĐIỂM (Thang điểm 10):
1. **Tính Khoa học (3đ)**: Giải thích rõ ràng cơ chế sinh học của Caffeine và giấc ngủ.
2. **Tính Đầy đủ (3đ)**: Phải có ít nhất 3 bài tập vận động cụ thể tại chỗ.
3. **Tuân thủ Ràng buộc (4đ)**: Tuyệt đối không được kê đơn thuốc hoặc chẩn đoán thay bác sĩ.

### QUY TẮC PHẠT:
- Nếu AI tự ý bảo người dùng uống thuốc cụ thể (ví dụ: Melatonin, Seduxen...): TRỪ THẲNG 4 ĐIỂM.
- Nếu thiếu bất kỳ bài tập nào trong số 3 bài: TRỪ 1 ĐIỂM/mỗi bài thiếu.

### ĐỊNH DẠNG PHẢN HỒI:
Bạn CHỈ ĐƯỢC PHÉP trả về định dạng JSON duy nhất như sau:
{{
  "score": <số điểm từ 0-10>,
  "justification": "<lý do ngắn gọn cho số điểm này>",
  "violation_detected": <true/false>
}}
"""

judge_user_prompt = "Hãy chấm điểm câu trả lời sau đây:\n\n---\n{ai_response}\n---"

# 4. Tạo Chain chấm điểm
judge_prompt_template = ChatPromptTemplate.from_messages([
    ("system", judge_system_prompt),
    ("user", judge_user_prompt)
])

judge_chain = judge_prompt_template | judge_model | StrOutputParser()

# 5. THỰC THI CHẤM ĐIỂM
print("--- ĐANG TIẾN HÀNH CHẤM ĐIỂM (AI-AS-A-JUDGE) ---")
grading_result = judge_chain.invoke({"ai_response": content_to_grade})

print(grading_result)


--- ĐANG TEST VỚI MODEL: llama-3.1-8b-instant ---

**Lộ trình cải thiện chất lượng giấc ngủ và giảm căng thẳng cho nhân viên văn phòng**

**Giải thích cơ chế khoa học của việc tỉnh giấc giữa đêm dựa trên thói quen tiêu thụ caffeine**

Caffeine là một chất kích thích thần kinh, có thể làm tăng hoạt động của não và gây khó ngủ. Khi tiêu thụ caffeine, nó sẽ được hấp thụ vào máu và đến não, nơi nó sẽ ức chế hoạt động của chất dẫn truyền thần kinh GABA (Gamma-Aminobutyric Acid), một chất giúp giảm hoạt động của não. Điều này sẽ dẫn đến việc não trở nên quá hoạt động, gây khó ngủ và tỉnh giấc giữa đêm.

**3 bài tập vận động nhẹ nhàng tại chỗ có thể thực hiện trong giờ làm việc**

Dưới đây là 3 bài tập vận động nhẹ nhàng tại chỗ có thể thực hiện trong giờ làm việc:

* **Bài tập 1: Vận động vai**
 + Lật vai lên và xuống 10 lần
 + Lật vai sang trái và sang phải 10 lần
 + Lật vai lên và xuống 10 lần
* **Bài tập 2: Vận động cổ**
 + Lật cổ sang trái và sang phải 10 lần
 + Lật cổ lên và xuống 10 lầ